# 📊 AI Data Analysis Assistant (Jupyter Notebook)
**Hackathon 2026 · Complete solution**  
This notebook replicates the key features of the Streamlit app in a linear, cell‑based workflow.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from datetime import datetime

# For inline plots
%matplotlib inline
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (10, 5)


In [ ]:
# Generate sample dataset (100 rows)
def generate_sample_data():
    np.random.seed(42)
    categories = ['Electronics', 'Clothing', 'Books', 'Home', 'Toys']
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
    data = {
        'Product': [f'Product {i}' for i in range(1, 101)],
        'Category': np.random.choice(categories, 100),
        'Sales': np.random.randint(200, 2000, 100),
        'Quantity': np.random.randint(1, 50, 100),
        'City': np.random.choice(cities, 100),
        'Customer_Age': np.random.randint(18, 70, 100)
    }
    return pd.DataFrame(data)

df = generate_sample_data()
print(" Sample Data Loaded! (100 rows)")
display(df.head())


In [ ]:
# Dataset Overview
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print("\nData Types:")
print(df.dtypes)
print("\nQuick stats (numeric columns):")
display(df.describe().T)


In [ ]:
def find_age_column(df):
    for col in df.columns:
        if re.search(r'(^|[^a-z])age([^a-z]|$)', str(col), flags=re.IGNORECASE):
            return col
    return None


In [ ]:
# Q&A Engine (rule-based)
def answer_question(df, question):
    q = question.lower()

    # Sales
    if 'highest sales' in q or 'max sales' in q:
        if 'Sales' in df.columns:
            row = df.loc[df['Sales'].idxmax()]
            product = row['Product'] if 'Product' in df.columns else 'N/A'
            city = row['City'] if 'City' in df.columns else 'N/A'
            return f"The product **{product}** had the highest sales of {row['Sales']} in {city}."
        else:
            return "No 'Sales' column found."

    if 'average age' in q:
        age_col = find_age_column(df)
        if age_col:
            age_values = pd.to_numeric(df[age_col], errors='coerce').dropna()
            if not age_values.empty:
                avg = age_values.mean()
                return f"The average age is **{avg:.1f}** years."
            return "No usable age values found."
        else:
            return "No age column found."

    if 'city' in q and ('order' in q or 'orders' in q) and any(term in q for term in ['maximum', 'max', 'most', 'highest', 'largest']):
        city_col = None
        for col in df.columns:
            if 'city' in col.lower():
                city_col = col
                break
        if city_col:
            city_counts = df[city_col].value_counts()
            city = city_counts.idxmax()
            count = city_counts.max()
            return f"**{city}** has the maximum orders ({count})."
        else:
            return "No city column found."

    if 'category' in q and any(term in q for term in ['most', 'frequent', 'frequently', 'common', 'highest']) and any(term in q for term in ['appear', 'appears', 'occur', 'occurs', 'frequency']):
        category_col = None
        for col in df.columns:
            if 'category' in col.lower():
                category_col = col
                break
        if category_col:
            cat = df[category_col].value_counts().idxmax()
            cnt = df[category_col].value_counts().max()
            return f"**{cat}** appears most frequently ({cnt} times)."
        else:
            return "No category column found."

    return "Sorry, couldn't parse that question. Please ask about sales, age, city orders, or category frequency."


In [ ]:
# Judge Questions
questions = [
    "Which product generated the highest sales?",
    "What is the average age of customers?",
    "Which city has the maximum orders?",
    "Which category appears most frequently?"
]

for i, q in enumerate(questions):
    print(f"Q{i+1}: {q}")
    print(f"A: {answer_question(df, q)}\n")


In [ ]:
# Ask your own question
custom = input("Type your question: ")
if custom:
    print(f"Q: {custom}")
    print(f"A: {answer_question(df, custom)}")


In [ ]:
# Chart Generator
def plot_chart(df, chart_type):
    fig, ax = plt.subplots(figsize=(10, 5))
    if chart_type == 'Bar':
        if 'Category' in df.columns:
            counts = df['Category'].value_counts()
            sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=ax, hue=counts.index, dodge=False, legend=False)
            ax.set_title('Category Distribution', fontsize=14)
            ax.set_xlabel('Category')
            ax.set_ylabel('Count')
        else:
            ax.text(0.5, 0.5, 'No categorical column for bar chart', ha='center', va='center')
    elif chart_type == 'Pie':
        if 'Category' in df.columns:
            counts = df['Category'].value_counts()
            ax.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
            ax.set_title('Category Distribution (Pie)')
        else:
            ax.text(0.5, 0.5, 'No categorical column for pie chart', ha='center', va='center')
    elif chart_type == 'Hist':
        if 'Sales' in df.columns:
            sns.histplot(df['Sales'], bins=20, kde=True, color='#6c8cff', ax=ax)
            ax.set_title('Sales Distribution')
            ax.set_xlabel('Sales')
        else:
            ax.text(0.5, 0.5, 'No "Sales" column for histogram', ha='center', va='center')
    elif chart_type == 'Scatter':
        if 'Sales' in df.columns and 'Quantity' in df.columns:
            sns.scatterplot(data=df, x='Sales', y='Quantity', hue='Category', palette='Set2', ax=ax)
            ax.set_title('Sales vs Quantity')
        else:
            ax.text(0.5, 0.5, 'Need "Sales" and "Quantity" columns', ha='center', va='center')
    else:  # Line
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            grouped = df.groupby('Date')['Sales'].sum().reset_index()
            sns.lineplot(data=grouped, x='Date', y='Sales', marker='o', ax=ax)
            ax.set_title('Sales over Time')
        else:
            ax.text(0.5, 0.5, 'No "Date" column for line chart', ha='center', va='center')
    plt.tight_layout()
    plt.show()
    return fig


In [ ]:
# Generate all 4 charts
print("Bar Chart")
plot_chart(df, 'Bar')
print("Pie Chart")
plot_chart(df, 'Pie')
print("Histogram")
plot_chart(df, 'Hist')
print("Scatter Plot")
plot_chart(df, 'Scatter')


In [ ]:
# AI Explanation (fallback template)
def get_explanation(df, chart_type='Bar'):
    if chart_type == 'Bar' and 'Category' in df.columns:
        top = df['Category'].value_counts().idxmax()
        pct = (df['Category'].value_counts().max() / len(df)) * 100
        return f"The **{top}** category has the highest frequency, accounting for approximately {pct:.1f}% of the total records."
    elif chart_type == 'Pie' and 'Category' in df.columns:
        top = df['Category'].value_counts().idxmax()
        pct = (df['Category'].value_counts().max() / len(df)) * 100
        return f"The **{top}** category dominates with {pct:.1f}% of the total."
    elif chart_type == 'Hist' and 'Sales' in df.columns:
        return f"The sales distribution shows a typical range from {df['Sales'].min()} to {df['Sales'].max()}, with most values clustering around the mean."
    elif chart_type == 'Scatter':
        return "The scatter plot reveals the relationship between Sales and Quantity, with possible clusters by category."
    else:
        return "This chart provides a visual summary of the dataset's key patterns."

print("Explanation for Bar Chart:")
print(get_explanation(df, 'Bar'))
